# Common Functions used:

## 1) Add Data Source and Date

In [0]:
import datetime

today_str = datetime.date.today().isoformat()

dbutils.widgets.text("p_data_source", "Jolpica F1 API")
dbutils.widgets.text("p_file_date", today_str)

v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")

## 2) Add Ingestion Date:

In [0]:
from pyspark.sql.functions import current_timestamp

def add_ingestion_date(input_df):
    return input_df.withColumn("ingestion_date", current_timestamp())

## 3) Add Surrogate Key:

In [0]:
from pyspark.sql.functions import sha2, concat_ws, col


def add_surrogate_key(input_df, key_column_name, hash_columns):
    return input_df.withColumn(
        key_column_name,
        sha2(concat_ws("||", *[col(c).cast("string") for c in hash_columns]), 256)
    )

## 4) Upsert New Data if changed (for non-partition and partitioned Delta Lake):

In [0]:
def upsert_if_changed(input_df, db_name, table_name, output_path, merge_key_columns):
    from delta.tables import DeltaTable

    dbutils.fs.mkdirs(output_path)

    table_registered = spark.catalog.tableExists(f"{db_name}.{table_name}")

    if not table_registered:
        input_df.write.mode("overwrite").format("delta").save(output_path)
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {db_name}.{table_name}
            USING DELTA
            LOCATION '{output_path}'
        """)
        print(f"Created table {db_name}.{table_name} with {input_df.count()} row(s)")
        return

    if not DeltaTable.isDeltaTable(spark, output_path):
        raise Exception(
            f"{db_name}.{table_name} is registered in the metastore but '{output_path}' "
            f"is not a valid Delta table. Fix or drop the table manually before continuing."
        )

    sk_columns = [c for c in input_df.columns if c.endswith("_sk")]
    if not sk_columns:
        raise Exception(f"No '_sk' surrogate key column found in input_df columns: {input_df.columns}")
    key_column = sk_columns[0]

    delta_table = DeltaTable.forPath(spark, output_path)

    rows_before = delta_table.toDF().count()

    delta_table.alias("tgt") \
        .merge(input_df.alias("src"), f"tgt.{key_column} = src.{key_column}") \
        .whenNotMatchedInsertAll() \
        .execute()

    rows_after = delta_table.toDF().count()
    rows_inserted = rows_after - rows_before

    print(f"{db_name}.{table_name}: {rows_inserted} new row(s) inserted "
          f"(rows before: {rows_before}, rows after: {rows_after})")

In [0]:
def upsert_if_changed_for_seasons(input_df, db_name, table_name, output_path, merge_key_columns, partition_columns):
    from delta.tables import DeltaTable

    dbutils.fs.mkdirs(output_path)

    table_registered = spark.catalog.tableExists(f"{db_name}.{table_name}")

    if not table_registered:
        input_df.write.mode("overwrite").format("delta").partitionBy(*partition_columns).save(output_path)
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {db_name}.{table_name}
            USING DELTA
            LOCATION '{output_path}'
        """)
        print(f"Created table {db_name}.{table_name} with {input_df.count()} row(s), partitioned by {partition_columns}")
        return

    if not DeltaTable.isDeltaTable(spark, output_path):
        raise Exception(
            f"{db_name}.{table_name} is registered in the metastore but '{output_path}' "
            f"is not a valid Delta table. Fix or drop the table manually before continuing."
        )

    sk_columns = [c for c in input_df.columns if c.endswith("_sk")]
    if not sk_columns:
        raise Exception(f"No '_sk' surrogate key column found in input_df columns: {input_df.columns}")
    key_column = sk_columns[0]

    delta_table = DeltaTable.forPath(spark, output_path)

    rows_before = delta_table.toDF().count()

    delta_table.alias("tgt") \
        .merge(input_df.alias("src"), f"tgt.{key_column} = src.{key_column}") \
        .whenNotMatchedInsertAll() \
        .execute()

    rows_after = delta_table.toDF().count()
    rows_inserted = rows_after - rows_before

    print(f"{db_name}.{table_name}: {rows_inserted} new row(s) inserted "
          f"(rows before: {rows_before}, rows after: {rows_after})")

## 5) Build presentation dimension and fact tables:

In [0]:
def build_presentation_dimension(processed_location, natural_key_column, keep_columns, presentation_directory, db_name, table_name):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, desc

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    processed_df = spark.read.format("delta").load(processed_location)

    latest_version_window = Window.partitionBy(natural_key_column).orderBy(desc("ingestion_date"))

    deduped_df = processed_df \
        .withColumn("row_num", row_number().over(latest_version_window)) \
        .filter("row_num = 1") \
        .select(keep_columns)

    dbutils.fs.mkdirs(presentation_directory)

    deduped_df.write.mode("overwrite").format("delta").save(presentation_directory)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name}
        USING DELTA
        LOCATION '{presentation_directory}'
    """)

    print(f"Wrote {deduped_df.count()} row(s) to {db_name}.{table_name} ({presentation_directory})")
    return deduped_df

In [0]:
def build_presentation_fact(processed_location, presentation_directory, db_name, table_name, partition_columns=None, drop_columns=None):
    if drop_columns is None:
        drop_columns = ["data_source", "file_date", "ingestion_date"]

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    processed_df = spark.read.format("delta").load(processed_location)

    sk_columns = [c for c in processed_df.columns if c.endswith("_sk")]
    columns_to_drop = [c for c in (drop_columns + sk_columns) if c in processed_df.columns]

    fact_df = processed_df.drop(*columns_to_drop)

    dbutils.fs.mkdirs(presentation_directory)

    writer = fact_df.write.mode("overwrite").format("delta")
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)
    writer.save(presentation_directory)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name}
        USING DELTA
        LOCATION '{presentation_directory}'
    """)

    print(f"Wrote {fact_df.count()} row(s) to {db_name}.{table_name} ({presentation_directory}), "
          f"dropped columns: {columns_to_drop}"
          + (f", partitioned by {partition_columns}" if partition_columns else ""))
    return fact_df